# Resumo e Decisões

## Parte 1:
Esses são os pontos que vou retirar, setembro de 2021, maio de 2024 e dezembro de 2025. Os dois primeiros baseados na média geral e o último baseado na média do respectivo mês nos últimos 4 anos

## Parte 2:
Removido os outliers e feita a média de geração de cada mês. No ponto de outlier foi alterado o valor para a média do mês nos 4 anos disponíveis. Dessa forma evitamos problemas na etapa seguinte com modelos que precisam de dados contínuos.

### Parte 1

Importar as bibliotecas e carregar os dados de geração

In [ ]:
import pandas as pd

# 1. Carregar os dados
tabela_usina = pd.read_excel('power_plant_data_teste.xlsx')

# verificar se tem linhas vazias
display(tabela_usina.info())

# Mostrar a tabela
display(tabela_usina)

: 

In [ ]:
#Temos uma linha vazia na tabela! Limpando essa linha
tabela_limpa = tabela_usina.dropna()
display(tabela_limpa.info())

display(tabela_limpa.describe().round(1))

In [ ]:
import plotly.express as px

# Gráfico de barras simples: Mês no eixo X e Geração no eixo Y
fig = px.bar(
    tabela_usina, 
    x='Geração Mensal Referência Month', 
    y='Geração Mensal SUM Energia Gerada (kWh)',
    title='Geração Mensal da Usina (kWh)',
    labels={'data': 'Mês/Ano', 'geracao': 'Geração (kWh)'}
)

fig.show()


### Importante:
    Temos uma média de geração de 417105 kWh no período avaliado
    Setembro de 2022 tem a maior geração com 730252 kWh, aproximadamente 75% mais produção que a media
    Maio de 2024 tem a menor geração com 110463 kWh, aproximadamente 26% da média
    Dezembro de 2025 está com geração de 266 kWh, sendo aproximadamente 50% da média dos últimos 4 anos (21 a 24 => média aproximada 519 kwh)

Esses são os pontos que vou modificar: setembro de 2022, maio de 2024 e dezembro de 2025. Os dois primeiros baseados na média geral e o último baseado na média do respectivo mês nos últimos 4 anos


### Parte 2

In [ ]:
# variáveis auxiliares
col_data = 'Geração Mensal Referência Month'
col_geracao = 'Geração Mensal SUM Energia Gerada (kWh)'

# Converter a data
tabela_usina[col_data] = pd.to_datetime(tabela_usina[col_data])

# Definir os 3 pontos de outliers
datas_outliers = pd.to_datetime(['2022-09-01', '2024-05-01', '2025-12-01'])

# Criar coluna auxiliar com o mês do ano (1 a 12)
tabela_limpa['mes_num'] = tabela_limpa[col_data].dt.month

# ver se temos todas as linhas
display(tabela_limpa.info())

# mostrar tabela
#display(tabela_limpa)


In [ ]:
# Calcular a média do mês(excluindo os outliers)
dados_sem_outliers = tabela_limpa[~tabela_limpa[col_data].isin(datas_outliers)]
medias_historicas = dados_sem_outliers.groupby('mes_num')[col_geracao].mean()

# Criar a coluna tratada (inicialmente idêntica à original)
tabela_limpa['geracao_corrigida'] = tabela_limpa[col_geracao].astype(float)

# Substituir o valor dos 3 outliers pela média dos seus respectivos meses
for data in datas_outliers:
    mes = data.month
    valor_media = medias_historicas[mes]
    tabela_limpa.loc[tabela_limpa[col_data] == data, 'geracao_corrigida'] = valor_media

# Removendo a coluna auxiliar
tabela_limpa = tabela_limpa.drop(columns=['mes_num'])

# --- Relatório do Antes e Depois ---
print("=== RELATÓRIO DE CORREÇÃO (PARTE 2) ===")
filtro_outliers = tabela_limpa[col_data].isin(datas_outliers)
relatorio = tabela_limpa.loc[filtro_outliers, [col_data, col_geracao, 'geracao_corrigida']]
relatorio.columns = ['Data', 'Valor Original (Outlier)', 'Valor Corrigido (Média dos 4 anos)']

display(relatorio)

display(tabela_limpa.info())

In [ ]:
# Plotando o gráficos antes e depois para verificar os dados removidos

# antes
fig = px.bar(
    tabela_usina, 
    x='Geração Mensal Referência Month', 
    y='Geração Mensal SUM Energia Gerada (kWh)',
    title='Geração Mensal da Usina antes (kWh)',
    labels={'data': 'Mês/Ano', 'geracao': 'Geração (kWh)'}
)

fig.show()

# depois
fig = px.bar(
    tabela_limpa, 
    x='Geração Mensal Referência Month', 
    y='geracao_corrigida',
    title='Geração Mensal da Usina depois (kWh)',
    labels={'data': 'Mês/Ano', 'geracao': 'Geração (kWh)'}
)

fig.show()

Foram corrigidos os dados referentes a setembro de 2022, maio de 2024 e dezembro de 2025. Para todos os casos foram modificados seus valores para a média dos outros 4 meses disponíveis.